<a href="https://colab.research.google.com/github/jamie07262/INFO-3608-PROJECT/blob/main/models/KNN_Model_Full.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.metrics import accuracy_score, log_loss, make_scorer
from sklearn.pipeline import Pipeline

In [ ]:
# Define GitHub paths
github_url = "https://github.com/jamie07262/INFO-3608-PROJECT/tree/main/data/processed"
processed_url_prefix = "https://raw.githubusercontent.com/jamie07262/INFO-3608-PROJECT/main/data/processed/"

# Get the HTML content of the folder
response = requests.get(github_url)
html = response.text

# Extract CSV filenames and remove duplicates
csv_files = list(set(re.findall(r'href=".*?/data/processed/([^"]+\.csv)"', html)))

# Create the directory and download the files
!mkdir -p processed
for file in csv_files:
    url = processed_url_prefix + file
    !wget -q -P processed {url}
    print(f"Downloaded: {file}")

print("All CSV files downloaded successfully.")

In [ ]:
# Load the features file
df_features = pd.read_csv("processed/Features.csv")
display(df_features)

In [ ]:
import requests
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.metrics import accuracy_score, log_loss
from sklearn.pipeline import Pipeline

# Define all possible features
ALL_FEATURES = [
    "WinPercentage",
    "WinPercentageDifference",
    "538rating",
    "538ratingOpponent",
    "BaselinePred"
]
TARGET = "Win"

# Prepare data for modeling
x = df_features[ALL_FEATURES]
y = df_features[TARGET]
groups = df_features["Season"]
seasons = df_features["Season"].unique()

# Set up GroupKFold for cross-validation by season
gkf = GroupKFold(n_splits=len(seasons))

cv_results = []
best_models = []
season_best_feature_sets = []

# Cross-validation loop
for season_idx, (train_index, test_index) in enumerate(gkf.split(x, y, groups)):
    holdout_season = seasons[season_idx]
    x_train_full, x_test_full = x.iloc[train_index], x.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    groups_train = groups.iloc[train_index]

    print(f"\n=== Holdout Season: {holdout_season} ===")

    season_results = []
    best_cv_score = -np.inf
    best_combo_result = None

    # Try all non-empty combinations of features
    for r in range(1, len(ALL_FEATURES) + 1):
        for combo in combinations(ALL_FEATURES, r):
            feature_list = list(combo)
            x_train = x_train_full[feature_list]
            x_test = x_test_full[feature_list]

            pipeline = Pipeline([
                ('knn', KNeighborsClassifier())
            ])

            sample_size = x_train.shape[0]
            max_k = min(999, sample_size)
            param_grid = {
                'knn__n_neighbors': list(range(50, 200, 1)),
                'knn__weights': ['uniform', 'distance']
            }

            inner_cv = GroupKFold(n_splits=len(np.unique(groups_train)))
            grid_search = GridSearchCV(
                estimator=pipeline,
                param_grid=param_grid,
                scoring='accuracy',
                cv=inner_cv,
                n_jobs=-1
            )

            grid_search.fit(x_train, y_train, groups=groups_train)

            best_model = grid_search.best_estimator_

            y_prob = best_model.predict_proba(x_test)
            y_pred = best_model.predict(x_test)

            score_ll = log_loss(y_test, y_prob)
            accuracy = accuracy_score(y_test, y_pred)

            result = {
                "Season": holdout_season,
                "Accuracy": accuracy,
                "LogLoss": score_ll,
                "Best Params": grid_search.best_params_,
                "Features": feature_list,
                "CV Score": grid_search.best_score_,
                "Best Estimator": best_model
            }

            cv_results.append(result)
            season_results.append(result)

            print(f"Features: {feature_list} | Accuracy: {accuracy:.4f} | LogLoss: {score_ll:.4f} | CV Score: {grid_search.best_score_:.4f}")

            if grid_search.best_score_ > best_cv_score:
                best_cv_score = grid_search.best_score_
                best_combo_result = result

    # Store best for this season based on CV score
    season_best_feature_sets.append(best_combo_result)

    # Show top 5 models by CV score
    top5 = sorted(season_results, key=lambda x: x["CV Score"], reverse=True)[:5]
    df_top5 = pd.DataFrame(top5)[["Features", "CV Score", "Accuracy", "LogLoss", "Best Params"]]
    print(f"\n>>> Top 5 Feature Sets for {holdout_season} (by CV Score):")
    display(df_top5)

# Results DataFrames
df_cv_results = pd.DataFrame(cv_results)
df_best_by_season = pd.DataFrame(season_best_feature_sets)

# Display all results and best-per-season
display(df_cv_results)
display(df_best_by_season)

print(f"\nAverage Accuracy (All Combos): {df_cv_results['Accuracy'].mean():.4f}")
print(f"Average Log Loss (All Combos): {df_cv_results['LogLoss'].mean():.4f}")

# Plot best models per season
plt.figure(figsize=(10, 5))
plt.plot(df_best_by_season['Season'], df_best_by_season['Accuracy'], marker='o', label='Best Accuracy per Season')
plt.plot(df_best_by_season['Season'], df_best_by_season['LogLoss'], marker='o', label='Best Accuracy per Season')
plt.xlabel("Season")
plt.ylabel("Accuracy")
plt.title("Best Feature Set Accuracy by Season (Based on CV Best Estimator)")
plt.xticks(rotation=45)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import requests
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.metrics import accuracy_score, log_loss
from sklearn.pipeline import Pipeline

# Define all possible features
ALL_FEATURES = [
    "WinPercentage",
    "MedianScoreDifference",
    "HighSeed",
    "OpponentWinPercentage",
    "OpponentMedianScoreDifference",
    "OpponentHighSeed",
    "WinPercentageDifference",
    "HighSeedDifference",
    "MedianScoreDifferenceDifference",
    "538rating",
    "538ratingOpponent",
    "538rating_Difference",
    "BaselinePred"
]

TARGET = "Win"

# Prepare data for modeling
x = df_features[ALL_FEATURES]
y = df_features[TARGET]
groups = df_features["Season"]
seasons = df_features["Season"].unique()

# Set up GroupKFold for cross-validation by season
gkf = GroupKFold(n_splits=len(seasons))

cv_results = []
best_models = []
season_best_feature_sets = []

# Cross-validation loop
for season_idx, (train_index, test_index) in enumerate(gkf.split(x, y, groups)):
    holdout_season = seasons[season_idx]
    x_train_full, x_test_full = x.iloc[train_index], x.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    groups_train = groups.iloc[train_index]

    print(f"\n=== Holdout Season: {holdout_season} ===")

    season_results = []
    best_cv_score = -np.inf
    best_combo_result = None

    # Try all non-empty combinations of features
    for r in range(1, len(ALL_FEATURES) + 1):
        for combo in combinations(ALL_FEATURES, r):
            feature_list = list(combo)
            x_train = x_train_full[feature_list]
            x_test = x_test_full[feature_list]

            pipeline = Pipeline([
                ('knn', KNeighborsClassifier())
            ])

            sample_size = x_train.shape[0]
            max_k = min(999, sample_size)
            param_grid = {
                'knn__n_neighbors': list(range(75, 230, 1)),
                'knn__weights': ['uniform', 'distance']
            }

            inner_cv = GroupKFold(n_splits=len(np.unique(groups_train)))
            grid_search = GridSearchCV(
                estimator=pipeline,
                param_grid=param_grid,
                scoring='accuracy',
                cv=inner_cv,
                n_jobs=-1
            )

            grid_search.fit(x_train, y_train, groups=groups_train)

            best_model = grid_search.best_estimator_

            y_prob = best_model.predict_proba(x_test)
            y_pred = best_model.predict(x_test)

            score_ll = log_loss(y_test, y_prob)
            accuracy = accuracy_score(y_test, y_pred)

            result = {
                "Season": holdout_season,
                "Accuracy": accuracy,
                "LogLoss": score_ll,
                "Best Params": grid_search.best_params_,
                "Features": feature_list,
                "CV Score": grid_search.best_score_,
                "Best Estimator": best_model
            }

            cv_results.append(result)
            season_results.append(result)

            print(f"Features: {feature_list} | Accuracy: {accuracy:.4f} | LogLoss: {score_ll:.4f} | CV Score: {grid_search.best_score_:.4f}")

            if grid_search.best_score_ > best_cv_score:
                best_cv_score = grid_search.best_score_
                best_combo_result = result

    # Store best for this season based on CV score
    season_best_feature_sets.append(best_combo_result)

    # Show top 5 models by CV score
    top5 = sorted(season_results, key=lambda x: x["CV Score"], reverse=True)[:5]
    df_top5 = pd.DataFrame(top5)[["Features", "CV Score", "Accuracy", "LogLoss", "Best Params"]]
    print(f"\n>>> Top 5 Feature Sets for {holdout_season} (by CV Score):")
    display(df_top5)

# Results DataFrames
df_cv_results = pd.DataFrame(cv_results)
df_best_by_season = pd.DataFrame(season_best_feature_sets)

# Display all results and best-per-season
display(df_cv_results)
display(df_best_by_season)

print(f"\nAverage Accuracy (All Combos): {df_cv_results['Accuracy'].mean():.4f}")
print(f"Average Log Loss (All Combos): {df_cv_results['LogLoss'].mean():.4f}")

# Plot best models per season
plt.figure(figsize=(10, 5))
plt.plot(df_best_by_season['Season'], df_best_by_season['Accuracy'], marker='o', label='Best Accuracy per Season')
plt.xlabel("Season")
plt.ylabel("Accuracy")
plt.title("Best Feature Set Accuracy by Season (Based on CV Best Estimator)")
plt.xticks(rotation=45)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()
